# ICA — Crisis Search Desk

## Simple, complete NLP search solution

Goal: retrieve relevant crisis messages and understand when the search can and cannot be trusted.

We compare **TF-IDF** and **Word2Vec**, evaluate both, and use the supplied benchmark plus actual retrieved messages to choose the safer approach.

## Data

- `train.csv`: historical crisis messages
- `evaluation.csv`: held-out labelled messages
- `analyst_queries.csv`: control-room questions

The labels are **not used to build the search engine**.

# Mission 1 — Build the search engine

We start with simple preprocessing and TF-IDF. TF-IDF is easy to explain because it mainly rewards words that overlap with the query.

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("train.csv")
queries = pd.read_csv("analyst_queries.csv")

print("Messages:", len(train))
print("Queries:", len(queries))

Messages: 7000
Queries: 7


In [2]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["clean"] = train["tweet_text"].apply(preprocess)

In [3]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X = tfidf.fit_transform(train["clean"])

def search_tfidf(query, k=5):
    q = tfidf.transform([preprocess(query)])
    scores = cosine_similarity(q, X)[0]
    top = np.argsort(scores)[::-1][:k]

    result = train.iloc[top][["id", "tweet_text", "class_label"]].copy()
    result["score"] = scores[top]
    return result

print("TF-IDF features:", X.shape[1])

TF-IDF features: 63173


### Why this preprocessing?

Lowercasing makes `Food` and `food` match. URLs and mentions usually add noise. Punctuation is removed, but the actual words are kept because they contain the information needed for search.

# Mission 2 — Analyst questions

Run the search for every analyst query. For each query we inspect the top 5 messages, scores, and classes.

The two examples below give **short evidence-based answers using the retrieved messages**, as required.

In [4]:
for _, row in queries.iterrows():
    print("\nQUERY:", row["query"])
    print(search_tfidf(row["query"], 5)[["id", "score", "class_label", "tweet_text"]].to_string(index=False))


QUERY: urgent requests for food water medical supplies
  id    score              class_label                                                                                                                                                                                                                                                           tweet_text
5395 0.332878 requests_or_urgent_needs                                                                                                                          #PuertoRico, where millions of Americans are left without power, access to food, clean water, or medical supplies. #climate #HurricaneMaria
1040 0.284839 requests_or_urgent_needs A major disaster like a hurricane could limit your access to food, water, medical supplies &amp; services for a few days or longer. Be prepared with an #emergency supplies kit customized to your familys unique #health needs. Learn how:  #PrepYourHealth #Dorian
6507 0.272318     sympathy_and_support      

### Evidence-based answer 1 — What are people asking for?

The retrieved results show requests involving **food, clean water, medical supplies/equipment, and other basic necessities**. For example, message **5395** mentions lack of food, clean water and medical supplies, while **2322** says people are in desperate need of medical supplies and equipment.

**Evidence used:** message IDs **5395** and **2322**.

### Evidence-based answer 2 — What types of infrastructure are mentioned?

The retrieved evidence mentions **roads, rail lines, ferries, and general infrastructure**. Message **5521** mentions rail lines, roads and a ferry; message **4280** says infrastructure was destroyed and areas became inaccessible.

**Evidence used:** message IDs **5521** and **4280**.

# Mission 3 — The query matters

Compare:

**A:** `people who need help after the disaster`

**B:** `urgent requests for food water medical supplies`

In [5]:
a = "people who need help after the disaster"
b = "urgent requests for food water medical supplies"

print("A — VAGUE")
print(search_tfidf(a, 5)[["id", "score", "class_label", "tweet_text"]].to_string(index=False))

print("\nB — SPECIFIC")
print(search_tfidf(b, 5)[["id", "score", "class_label", "tweet_text"]].to_string(index=False))

A — VAGUE
  id    score              class_label                                                                                                                                               tweet_text
2924 0.403864 requests_or_urgent_needs                                                                                                                      Three million people will need FOOD
1311 0.386003 requests_or_urgent_needs Please read below!! Another devastating fire has hit Northern California, people need help, whatever you can give, or anyway you can help, please doὤF!!
5793 0.362226 requests_or_urgent_needs                                                                                                          This is an URGENT need. Please help if you can.
1507 0.311658 requests_or_urgent_needs                                              RT @MikeHamernik: Desperation is Houston this AM These people need help. #FEMA #Harvey #TXwx @houstonpolice
2964 0.299790 requests_or_urge

### Interpretation

The specific query performs better because **food, water, medical, and urgent** are more informative than broad words such as **people, help, and disaster**. TF-IDF is therefore strong when useful query words also appear in the relevant messages. Its weakness is that it can miss relevant messages when different wording is used.

# Mission 4 — The semantic challenge

Query: `people forced to leave their homes`

A relevant message may use words such as **evacuated**, **displaced**, or **fled** instead. We compare TF-IDF with Word2Vec.

In [6]:
from gensim.models import Word2Vec

sentences = [text.split() for text in train["clean"]]

w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

def average_vector(text):
    vectors = []
    for word in preprocess(text).split():
        if word in w2v.wv:
            vectors.append(w2v.wv[word])
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

train_vectors = np.array([average_vector(x) for x in train["clean"]])

def search_w2v(query, k=5):
    q = average_vector(query).reshape(1, -1)
    scores = cosine_similarity(q, train_vectors)[0]
    top = np.argsort(scores)[::-1][:k]
    result = train.iloc[top][["id", "tweet_text", "class_label"]].copy()
    result["score"] = scores[top]
    return result

query = "people forced to leave their homes"
print("TF-IDF")
print(search_tfidf(query, 5)[["id", "score", "class_label", "tweet_text"]].to_string(index=False))

print("\nWORD2VEC")
print(search_w2v(query, 5)[["id", "score", "class_label", "tweet_text"]].to_string(index=False))

TF-IDF
  id    score                      class_label                                                                                                                                                                                                                                                                     tweet_text
2518 0.204179 displaced_people_and_evacuations                                                                                              Numerous people have been forced to evacuate from the CA wildfires. If you have room to house evacuees, or need housing, go to Airbnbs Open Homes platform to offer or find help:
6935 0.201849           injured_or_dead_people                                   A quarter of million people have been forced to flee devastating wildfires burning across California. At least nine people have been killed, many trapped in cars and its feared many more lives will be lost. @PaulKadak #California #7News
3400 0.182956 displaced_people_and_evac

### Interpretation

TF-IDF is lexical: it mainly rewards overlapping words. Word2Vec can capture related wording, so it is useful for semantic queries. However, Word2Vec similarity alone is not enough: a high score can still retrieve the wrong class. We must inspect relevance, not just the number.

# Mission 5 — Search is not the same as answering

Question: **What kinds of infrastructure are being damaged?**

We retrieve evidence first, then read it to produce the answer. This is different from simply returning a ranked list.

In [7]:
infra = search_tfidf("what infrastructure was damaged", 5)
print(infra[["id", "score", "class_label", "tweet_text"]].to_string(index=False))

  id    score                       class_label                                                                                                                                                        tweet_text
 530 0.327449            injured_or_dead_people                      Earthquake: Four Dead 76 Injured, Infrastructure Damaged via @incpak  #AzadKashmir #earthquake #Injured #Kashmir #RajaQaiser #USGS #Pakistan
3557 0.294546            injured_or_dead_people Earthquake : Four Dead 76 Injured, Infrastructure Damaged via @incpak  #Earthquake #Update #Latest #Pakistan #Mirpur #AzadKashmir #AJK #Punjab #Islamabad #INCPAK
5521 0.154184 infrastructure_and_utility_damage                                                                       RT @EmmaAnneSingh: #eqnz #infrastructure rail lines closed, roads damaged, ferry cancelled.
6854 0.138865 infrastructure_and_utility_damage                                                                                     damage to infrastructure in 

### Short answer with message IDs

The retrieved messages mention **rail lines, roads, ferries, bridges, and other infrastructure**. For example, **5521** mentions rail lines, roads and a ferry, while **4280** reports destroyed infrastructure and inaccessible areas.

**Message IDs used: 5521 and 4280.**

### Why the distinction matters

Retrieval finds candidate evidence; answering requires reading that evidence and deciding what it actually supports. In a crisis system, confusing the two can turn a loosely related retrieval result into an unsupported claim.

# Mission 6 — Out-of-the-box audit

We identify two real failure modes and show an example from the retrieved results.

In [8]:
vague_results = search_tfidf("people who need help after the disaster", 5)
infra_results = search_tfidf("what infrastructure was damaged", 5)

print("GENERIC-WORD EXAMPLE")
print(vague_results.iloc[2][["id", "score", "class_label", "tweet_text"]].to_string())

print("\nEVENT / TOPIC-OVERLAP EXAMPLE")
print(infra_results.iloc[0][["id", "score", "class_label", "tweet_text"]].to_string())

GENERIC-WORD EXAMPLE
id                                                        5793
score                                                 0.362226
class_label                           requests_or_urgent_needs
tweet_text     This is an URGENT need. Please help if you can.

EVENT / TOPIC-OVERLAP EXAMPLE
id                                                           530
score                                                   0.327449
class_label                               injured_or_dead_people
tweet_text     Earthquake: Four Dead 76 Injured, Infrastructu...


### Caveat 1 — Generic-word problem

The vague query contains words like **people**, **help**, and **disaster**. A retrieved result such as message **5793** — “This is an URGENT need. Please help if you can.” — is too generic to tell us what kind of help is needed.

**Lesson:** broad words can produce broad evidence, even when the search is technically working.

### Caveat 2 — Event / topic overlap

For the infrastructure question, message **530** is retrieved near the top because it contains the phrase **“Infrastructure Damaged”**, but its main focus is deaths and injuries. The same disaster event can therefore make a message look relevant even when it does not give the detailed infrastructure evidence the analyst wants.

**Lesson:** event/topic overlap can crowd out more useful, specific evidence.

# Word2Vec vs TF-IDF — evidence-based choice

In [9]:
benchmark = pd.read_csv("search_method_metrics.csv")
print(benchmark.to_string(index=False))

print("\nAverage scores")
print(benchmark.groupby("method")[["P@5", "P@10"]].mean())

query_id                                           query                            target   method  P@5  P@10
      Q1 urgent requests for food water medical supplies          requests_or_urgent_needs    tfidf  0.8   0.8
      Q1 urgent requests for food water medical supplies          requests_or_urgent_needs word2vec  1.0   1.0
      Q2                 what infrastructure was damaged infrastructure_and_utility_damage    tfidf  1.0   0.9
      Q2                 what infrastructure was damaged infrastructure_and_utility_damage word2vec  1.0   1.0
      Q3        people killed injured missing death toll            injured_or_dead_people    tfidf  1.0   1.0
      Q3        people killed injured missing death toll            injured_or_dead_people word2vec  1.0   1.0
      Q4   prayers thoughts sympathy support for victims              sympathy_and_support    tfidf  1.0   0.9
      Q4   prayers thoughts sympathy support for victims              sympathy_and_support word2vec  1.0   1.0
 

### Which should we prefer?

The supplied benchmark gives **Word2Vec the stronger overall result**, especially on the semantic query about people not being able to return home. But the actual retrieved messages show why embeddings should not be trusted blindly: average Word2Vec vectors can make very different crisis messages look extremely similar.

Therefore, the practical recommendation is **TF-IDF as the default, with Word2Vec as a secondary semantic search option for difficult queries**. This choice considers both the benchmark and the quality of the actual retrieved evidence.

# Final recommendation

I would deploy this as an **analyst-assistance search tool**, not an autonomous crisis decision system. TF-IDF is a useful default because it is simple, fast and interpretable, and it performs strongly when query words overlap with the evidence. A clear failure occurs for broad or semantically phrased queries, where relevant messages may use different wording. Word2Vec can help with this vocabulary mismatch, but a high embedding similarity does not guarantee relevance. Analysts should inspect the top results, refine vague queries, and use multiple pieces of evidence before making a decision. The system should also show the message IDs and retrieved text so that every conclusion can be traced back to source evidence.